In [ ]:
import pickle
import numpy as np

with open('model/wesad/S2/S2.pkl', 'rb') as f:
    data = pickle.load(f, encoding='latin1')

print(type(data))  # <class 'dict'>
print(data.keys())  # dict_keys(['subject', 'signal', 'label'])

<class 'dict'>
dict_keys(['signal', 'label', 'subject'])


In [2]:
# Кто это?
print(data['subject'])  # S2

# Какие сигналы есть?
print(data['signal'].keys())  # dict_keys(['wrist', 'chest'])

# Сигналы с запястья
print(data['signal']['wrist'].keys())
# dict_keys(['ACC', 'BVP', 'EDA', 'TEMP'])

# Сигналы с груди
print(data['signal']['chest'].keys())
# dict_keys(['ACC', 'ECG', 'EDA', 'EMG', 'Resp', 'Temp'])

S2
dict_keys(['chest', 'wrist'])
dict_keys(['ACC', 'BVP', 'EDA', 'TEMP'])
dict_keys(['ACC', 'ECG', 'EMG', 'EDA', 'Temp', 'Resp'])


In [ ]:
# ECG - электрокардиограмма (самый важный!)
ecg = data['signal']['chest']['ECG']
print(f"ECG shape: {ecg.shape}")
# ECG shape: (180000, 1)

# Превратим в 1D массив
ecg = ecg.flatten()
print(f"ECG flatten: {ecg.shape}")
# ECG flatten: (180000,)

# Это сколько секунд?
duration = len(ecg) / 700  # 700 Hz - частота
print(f"Длительность: {duration:.1f} сек = {duration/60:.1f} минут")


ECG shape: (4255300, 1)
ECG flatten: (4255300,)
Длительность: 6079.0 сек = 101.3 минут


In [5]:
# Первые 700 отсчётов = первая секунда
first_second = ecg[:700]
print(first_second)
# [-0.09375  -0.09357  -0.09339 ... -0.04236  -0.04309]

# Статистика
print(f"Mean: {np.mean(ecg):.4f}")
print(f"Std:  {np.std(ecg):.4f}")
print(f"Min:  {np.min(ecg):.4f}")
print(f"Max:  {np.max(ecg):.4f}")


[ 2.14233398e-02  2.03247070e-02  1.65252686e-02  1.67083740e-02
  1.16729736e-02  4.89807129e-03  2.79235840e-03  6.08825684e-03
  9.93347168e-03  1.03912354e-02  6.82067871e-03  3.25012207e-03
 -1.05285645e-03 -7.46154785e-03 -1.37329102e-02 -1.66168213e-02
 -1.89971924e-02 -2.11944580e-02 -2.23846436e-02 -2.35748291e-02
 -2.17437744e-02 -1.66168213e-02 -8.65173340e-03  7.78198242e-04
  7.64465332e-03  1.44653320e-02  2.00958252e-02  2.44903564e-02
  2.78778076e-02  3.31878662e-02  3.78570557e-02  4.19769287e-02
  4.41284180e-02  4.57305908e-02  4.66461182e-02  4.63256836e-02
  4.75616455e-02  5.03997803e-02  5.29632568e-02  5.30548096e-02
  5.46112061e-02  5.74493408e-02  6.02874756e-02  6.10198975e-02
  6.50024414e-02  7.32879639e-02  7.85980225e-02  7.64465332e-02
  6.56890869e-02  5.32836914e-02  4.52728271e-02  4.27093506e-02
  4.28009033e-02  4.59594727e-02  4.63256836e-02  4.47692871e-02
  4.28924561e-02  4.17022705e-02  3.83148193e-02  3.69415283e-02
  4.42657471e-02  5.31005

In [6]:
labels = data['label'].flatten()
print(f"Labels shape: {labels.shape}")  # (180000,)

# Какие метки есть?
print(np.unique(labels))  # [0 1 2 3 4]

# Расшифровка
emotions = {
    0: 'Transient (переход)',
    1: 'Baseline (спокойствие)',
    2: 'Stress (стресс)',
    3: 'Amusement (веселье)',
    4: 'Meditation (медитация)'
}

# Сколько каждого?
for emotion_id in [1, 2, 3, 4]:
    count = np.sum(labels == emotion_id)
    pct = 100 * count / len(labels)
    duration = count / 700
    print(f"{emotions[emotion_id]:25s}: {count:6d} ({pct:5.1f}%) = {duration:6.1f} sec")


Labels shape: (4255300,)
[0 1 2 3 4 6 7]
Baseline (спокойствие)   : 800800 ( 18.8%) = 1144.0 sec
Stress (стресс)          : 430500 ( 10.1%) =  615.0 sec
Amusement (веселье)      : 253400 (  6.0%) =  362.0 sec
Meditation (медитация)   : 537599 ( 12.6%) =  768.0 sec


In [7]:
# Маска для baseline состояния
baseline_mask = (labels == 1)

# Применим маску к ECG
ecg_baseline = ecg[baseline_mask]
print(f"ECG baseline shape: {ecg_baseline.shape}")  # (25200,)
print(f"Duration: {len(ecg_baseline) / 700:.1f} sec")  # Duration: 36.0 sec

# Статистика для baseline
print(f"Baseline ECG Mean: {np.mean(ecg_baseline):.4f}")
print(f"Baseline ECG Std:  {np.std(ecg_baseline):.4f}")


ECG baseline shape: (800800,)
Duration: 1144.0 sec
Baseline ECG Mean: 0.0012
Baseline ECG Std:  0.1485


In [8]:
# Маска для stress
stress_mask = (labels == 2)
ecg_stress = ecg[stress_mask]

print(f"Stress ECG Mean: {np.mean(ecg_stress):.4f}")
print(f"Stress ECG Std:  {np.std(ecg_stress):.4f}")

# Сравнение
print(f"Baseline vs Stress:")
print(f"  Mean difference: {np.mean(ecg_baseline) - np.mean(ecg_stress):.4f}")
print(f"  Std difference:  {np.std(ecg_baseline) - np.std(ecg_stress):.4f}")


Stress ECG Mean: 0.0012
Stress ECG Std:  0.1324
Baseline vs Stress:
  Mean difference: 0.0000
  Std difference:  0.0161


In [9]:
# BVP - фотоплетизмограмма
bvp = data['signal']['wrist']['BVP'].flatten()
print(f"BVP shape: {bvp.shape}")  # (48000,)

# BVP на 64 Hz, поэтому:
duration = len(bvp) / 64
print(f"Duration: {duration:.1f} sec")  # Duration: 750.0 sec (почти все записи)

# Статистика
print(f"BVP Mean: {np.mean(bvp):.4f}")
print(f"BVP Std:  {np.std(bvp):.4f}")


BVP shape: (389056,)
Duration: 6079.0 sec
BVP Mean: -0.0004
BVP Std:  75.8712
